# Sales Forecasting for a Retail Chain

Major Project – forecast daily sales using historical data, lag features, and a RandomForestRegressor.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt

df = pd.read_csv('retail_sales.csv', parse_dates=['date'])
df = df.sort_values('date')
df.head()

In [ ]:
daily = df.groupby('date', as_index=False)['sales'].sum().sort_values('date')
daily['sales_lag1'] = daily['sales'].shift(1)
daily['sales_lag7'] = daily['sales'].shift(7)
daily = daily.dropna().reset_index(drop=True)

X = daily[['sales_lag1', 'sales_lag7']]
y = daily['sales']

split_idx = int(len(daily) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
dates_test = daily['date'].iloc[split_idx:]

y_pred_naive = X_test['sales_lag1'].values
mae_naive = mean_absolute_error(y_test, y_pred_naive)
mae_naive

In [ ]:
model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mae_model = mean_absolute_error(y_test, y_pred)
mae_model, mae_naive - mae_model

In [ ]:
N = min(60, len(y_test))
plt.figure(figsize=(10, 4))
plt.plot(dates_test.iloc[-N:], y_test.iloc[-N:], label='Actual')
plt.plot(dates_test.iloc[-N:], y_pred[-N:], label='Predicted')
plt.title('Actual vs Predicted Sales (Last 60 Days)')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.tight_layout()
plt.show()